# Laboratorium 4 — ETL w hurtowniach danych

Celem zadania jest przygotowanie pipeline ETL dla danych sprzedażowych Online Retail. Proces ETL składa się z trzech etapów: Extract, Transform oraz Load.

W zadaniu dane zostaną wczytane do Pandas, oczyszczone, wzbogacone o dodatkowe kolumny analityczne, a następnie zapisane jako tabela faktów `fact_sales.csv`.

In [2]:
import pandas as pd
from pathlib import Path

## Extract — wczytanie danych

W etapie Extract dane są pobierane ze źródła i wczytywane do środowiska analitycznego. W tym zadaniu źródłem danych jest plik `Online_Retail.csv`.

In [3]:
DATA_PATH = Path("data/Online_Retail.csv")
OUTPUT_PATH = Path("output")
OUTPUT_PATH.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH, encoding="latin1")

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 140227 entries, 0 to 140226
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    140227 non-null  str    
 1   StockCode    140227 non-null  str    
 2   Description  139769 non-null  str    
 3   Quantity     140227 non-null  int64  
 4   InvoiceDate  140227 non-null  str    
 5   UnitPrice    140227 non-null  float64
 6   CustomerID   95724 non-null   float64
 7   Country      140227 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 8.6 MB


In [5]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,140227.000000,140227.000000,95724.000000
mean,9.014940,5.154270,15292.283670
std,285.621873,105.272844,1736.617656
min,-74215.000000,0.000000,12346.000000
25%,1.000000,1.250000,13869.000000
50%,3.000000,2.460000,15194.000000
75%,10.000000,4.210000,16873.000000
max,74215.000000,16888.020000,18283.000000


In [6]:
df.isna().sum()

InvoiceNo          0
StockCode          0
Description      458
Quantity           0
InvoiceDate        0
UnitPrice          0
CustomerID     44503
Country            0
dtype: int64

## Transform — czyszczenie danych

W etapie Transform dane są czyszczone i przygotowywane do dalszej analizy. Usunięto rekordy bez `CustomerID`, rekordy z niepoprawną ilością oraz rekordy z ujemną ceną. Usunięto również duplikaty i poprawiono typy danych.

Na etapie transformacji dodano kolumnę `Revenue`, ponieważ pozwala ona analizować wartość sprzedaży i przychód.

In [7]:
df_clean = df.copy()

df_clean = df_clean.dropna(subset=["CustomerID"])
df_clean = df_clean[df_clean["Quantity"] > 0]
df_clean = df_clean[df_clean["UnitPrice"] >= 0]
df_clean = df_clean.drop_duplicates()

df_clean["CustomerID"] = df_clean["CustomerID"].astype(int)
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]

df_clean.head()

C:\Users\dodom\AppData\Local\Temp\ipykernel_19788\3696838909.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [8]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 92127 entries, 0 to 140226
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   InvoiceNo    92127 non-null  str           
 1   StockCode    92127 non-null  str           
 2   Description  92127 non-null  str           
 3   Quantity     92127 non-null  int64         
 4   InvoiceDate  92127 non-null  datetime64[us]
 5   UnitPrice    92127 non-null  float64       
 6   CustomerID   92127 non-null  int64         
 7   Country      92127 non-null  str           
 8   Revenue      92127 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(4)
memory usage: 7.0 MB


## Transform — przetwarzanie dat

Kolumna `InvoiceDate` została rozbita na rok, miesiąc i dzień. Dzięki temu dane można łatwiej analizować w czasie, np. według miesięcy albo lat.

In [9]:
df_clean["Year"] = df_clean["InvoiceDate"].dt.year
df_clean["Month"] = df_clean["InvoiceDate"].dt.month
df_clean["Day"] = df_clean["InvoiceDate"].dt.day

df_clean[["InvoiceDate", "Year", "Month", "Day"]].head()

,InvoiceDate,Year,Month,Day
0,2010-12-01 08:26:00,2010,12,1
1,2010-12-01 08:26:00,2010,12,1
2,2010-12-01 08:26:00,2010,12,1
3,2010-12-01 08:26:00,2010,12,1
4,2010-12-01 08:26:00,2010,12,1


## Przygotowanie tabeli faktów

Tabela faktów zawiera dane sprzedażowe potrzebne do analiz. W tabeli pozostawiono numer faktury, kod produktu, identyfikator klienta, datę faktury, ilość oraz dodatkowe miary: cenę jednostkową i przychód.

In [10]:
fact_sales = df_clean[
    [
        "InvoiceNo",
        "StockCode",
        "CustomerID",
        "InvoiceDate",
        "Quantity",
        "UnitPrice",
        "Revenue",
        "Year",
        "Month",
        "Day",
        "Country"
    ]
].copy()

fact_sales.head()

,InvoiceNo,StockCode,CustomerID,InvoiceDate,Quantity,UnitPrice,Revenue,Year,Month,Day,Country
0,536365,85123A,17850,2010-12-01 08:26:00,6,2.55,15.30,2010,12,1,United Kingdom
1,536365,71053,17850,2010-12-01 08:26:00,6,3.39,20.34,2010,12,1,United Kingdom
2,536365,84406B,17850,2010-12-01 08:26:00,8,2.75,22.00,2010,12,1,United Kingdom
3,536365,84029G,17850,2010-12-01 08:26:00,6,3.39,20.34,2010,12,1,United Kingdom
4,536365,84029E,17850,2010-12-01 08:26:00,6,3.39,20.34,2010,12,1,United Kingdom


In [11]:
fact_sales.info()

<class 'pandas.DataFrame'>
Index: 92127 entries, 0 to 140226
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   InvoiceNo    92127 non-null  str           
 1   StockCode    92127 non-null  str           
 2   CustomerID   92127 non-null  int64         
 3   InvoiceDate  92127 non-null  datetime64[us]
 4   Quantity     92127 non-null  int64         
 5   UnitPrice    92127 non-null  float64       
 6   Revenue      92127 non-null  float64       
 7   Year         92127 non-null  int32         
 8   Month        92127 non-null  int32         
 9   Day          92127 non-null  int32         
 10  Country      92127 non-null  str           
dtypes: datetime64[us](1), float64(2), int32(3), int64(2), str(3)
memory usage: 7.4 MB


In [12]:
sales_by_country = (
    fact_sales
    .groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

sales_by_country.head(10)

,Country,Revenue
0,United Kingdom,1737612.64
1,Netherlands,80824.04
2,EIRE,62335.55
3,Germany,55743.34
4,France,44863.75
5,Australia,41902.47
6,Spain,19407.47
7,Japan,13534.78
8,Sweden,12550.70
9,Switzerland,10061.30


## Load — zapis danych

W etapie Load przygotowana tabela faktów zostaje zapisana do pliku CSV. Dzięki temu wynik procesu ETL może być wykorzystany w dalszych etapach budowy hurtowni danych albo w raportowaniu.

In [13]:
fact_sales.to_csv(OUTPUT_PATH / "fact_sales.csv", index=False)

print("Zapisano plik:", OUTPUT_PATH / "fact_sales.csv")

Zapisano plik: output\fact_sales.csv


In [14]:
list(OUTPUT_PATH.iterdir())

[WindowsPath('output/fact_sales.csv')]

## Podsumowanie

Zbudowano prosty pipeline ETL dla danych Online Retail. W etapie Extract dane zostały wczytane z pliku CSV, w etapie Transform zostały oczyszczone i wzbogacone o kolumny daty oraz miarę `Revenue`, a w etapie Load zapisano wynik jako `fact_sales.csv`.

Dodanie kolumny `Revenue` na etapie ETL ułatwia późniejszą analizę wartości sprzedaży. Przygotowana tabela faktów może być wykorzystana jako podstawa do dalszego modelowania hurtowni danych.